In [3]:
%pip install gradio redis pandas numpy sentence-transformers langchain langchain-groq python-dotenv ipykernel

Note: you may need to restart the kernel to use updated packages.


In [48]:
import os
import numpy as np
import pandas as pd
import time
import redis
import gradio as gr

from pathlib import Path
from dotenv import load_dotenv

from redis.commands.search.field import VectorField
from redis.commands.search.field import TextField
from redis.commands.search.field import TagField
from redis.commands.search.query import Query

from sentence_transformers import SentenceTransformer

from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.memory import ConversationBufferMemory
from langchain_groq import ChatGroq

# Load environment variables
load_dotenv()

# Redis connection
redis_conn = redis.Redis(
    host=os.getenv("REDIS_HOST"),
    port=int(os.getenv("REDIS_PORT")),
    decode_responses=False,
    username=os.getenv("REDIS_USERNAME", "default"),
    password=os.getenv("REDIS_PASSWORD"),
)

print("Redis Connected:", redis_conn.ping())

Redis Connected: True


In [49]:
MAX_TEXT_LENGTH = 512
NUMBER_PRODUCTS = 1000

def auto_truncate(val):
    return str(val)[:MAX_TEXT_LENGTH] if pd.notna(val) else ""

# Load product data and truncate long text fields
all_prods_df = pd.read_csv(
    "product_data.csv",
    converters={
        'bullet_point': auto_truncate,
        'item_keywords': auto_truncate,
        'item_name': auto_truncate
    }
)

all_prods_df['primary_key'] = (
    all_prods_df['item_id'].astype(str)
    + '-'
    + all_prods_df['domain_name'].astype(str)
)

all_prods_df['item_keywords'].replace('', np.nan, inplace=True)

all_prods_df.dropna(
    subset=['item_keywords'],
    inplace=True
)

all_prods_df.reset_index(drop=True, inplace=True)

# Get the first 1000 products with non-empty item keywords
product_metadata = all_prods_df.head(NUMBER_PRODUCTS).to_dict(
    orient='index'
)

all_prods_df.head()

C:\Users\ASUS\AppData\Local\Temp\ipykernel_13568\806800358.py:23: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  all_prods_df['item_keywords'].replace('', np.nan, inplace=True)


,item_id,marketplace,country,main_image_id,domain_name,bullet_point,item_keywords,material,brand,color,item_name,model_name,model_number,product_type,primary_key
0,B07T6RZ2CM,Amazon,IN,71dZhpsferL,amazon.in,3D Printed Hard Back Case Mobile Cover for Len...,mobile cover back cover mobile case phone case...,NaN,Amazon Brand - Solimo,Others,Amazon Brand - Solimo Designer Couples Sitting...,Lenovo K4 Note,gz8115-SL40423,CELLULAR_PHONE_CASE,B07T6RZ2CM-amazon.in
1,B07T2JY31Y,Amazon,IN,71vX7qIEAIL,amazon.in,3D Printed Hard Back Case Mobile Cover for Son...,mobile cover back cover mobile case phone case...,Wood,Amazon Brand - Solimo,others,Amazon Brand - Solimo Designer Leaf on Wood 3D...,Sony Xperia Z1 L39H,gz8056-SL40528,CELLULAR_PHONE_CASE,B07T2JY31Y-amazon.in
2,B0849YGSCZ,Amazon,AE,A1EZF-2mB5L,amazon.ae,,small de fur rooms navidad woven girls shag pa...,NaN,Stone & Beam,NaN,Stone & Beam Contemporary Doily Wool Farmhouse...,NaN,I59I8044IVYGRYC00-Parent,HOME_FURNITURE_AND_DECOR,B0849YGSCZ-amazon.ae
3,B081K6TCML,Amazon,IN,81o9EyZ-fAL,amazon.in,Solimo Plastic Multipurpose Modular Drawer; sm...,drawer modular drawer 3 rack modular drawer ki...,Plastic,Amazon Brand - Solimo,Multicolor,Amazon Brand - Solimo Plastic Multipurpose Mod...,NaN,sol_cujo_13,HOME,B081K6TCML-amazon.in
4,B0854774X5,Amazon,IN,81xaJCVnl3L,amazon.in,"Snug fit for Nokia 8.1, with perfect cut-outs ...",Back Cover Designer Case Designer Take It Easy...,Silicon,Amazon Brand - Solimo,Multicolor,Amazon Brand - Solimo Designer Take It Easy UV...,Nokia 8.1,UV10714-SL40617,CELLULAR_PHONE_CASE,B0854774X5-amazon.in


In [50]:
model = SentenceTransformer(
    'sentence-transformers/all-distilroberta-v1'
)

item_keywords = [
    product_metadata[i]['item_keywords']
    for i in product_metadata.keys()
]

item_keywords_vectors = [
    model.encode(sentence)
    for sentence in item_keywords
]

print("Embeddings generated successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings generated successfully!


In [51]:
len(item_keywords_vectors)
len(product_metadata)
product_metadata[0]

{'item_id': 'B07T6RZ2CM',
 'marketplace': 'Amazon',
 'country': 'IN',
 'main_image_id': '71dZhpsferL',
 'domain_name': 'amazon.in',
 'bullet_point': '3D Printed Hard Back Case Mobile Cover for Lenovo K4 Note Easy to put & take off with perfect cutouts for volume buttons, audio & charging ports. Stylish design and appearance, express your unique personality. Extreme precision design allows easy access to all buttons and ports while featuring raised bezel to life screen and camera off flat surface. Slim Hard Back Cover No Warranty None',
 'item_keywords': 'mobile cover back cover mobile case phone case mobile panel phone panel Lenovo mobile case Lenovo phone cover Lenovo back case hard case 3D printed mobile cover mobile cover back cover mobile case phone case mobile panel phone panel Lenovo mobile case Lenovo phone cover Lenovo back case hard case 3D printed mobile cover mobile cover back cover mobile case phone case mobile panel phone panel Lenovo mobile case Lenovo phone cover Lenovo 

In [52]:
def load_vectors(client, product_metadata, vector_dict, vector_field_name):

    p = client.pipeline(transaction=False)

    for index in product_metadata.keys():

        # Hash key
        key = 'product:' + str(index) + ':' + product_metadata[index]['primary_key']

        # Hash values
        item_metadata = product_metadata[index].copy()

        item_keywords_vector = vector_dict[index].astype(np.float32).tobytes()

        item_metadata[vector_field_name] = item_keywords_vector

        # Convert metadata values for Redis
        for field, value in item_metadata.items():

            if field != vector_field_name:
                if pd.isna(value):
                    item_metadata[field] = ""
                else:
                    item_metadata[field] = str(value)

        # HSET
        p.hset(key, mapping=item_metadata)

    p.execute()


def create_flat_index(
    redis_conn,
    vector_field_name,
    number_of_vectors,
    vector_dimensions=768,
    distance_metric='COSINE'
):

    redis_conn.ft().create_index([
        VectorField(
            vector_field_name,
            "FLAT",
            {
                "TYPE": "FLOAT32",
                "DIM": vector_dimensions,
                "DISTANCE_METRIC": distance_metric,
                "INITIAL_CAP": number_of_vectors,
                "BLOCK_SIZE": number_of_vectors
            }
        ),
        TagField("product_type"),
        TextField("item_name"),
        TextField("item_keywords"),
        TagField("country")
    ])

In [53]:
ITEM_KEYWORD_EMBEDDING_FIELD = 'item_keyword_vector'

TEXT_EMBEDDING_DIMENSION = 768

NUMBER_PRODUCTS = 1000

print('Loading and Indexing + ' + str(NUMBER_PRODUCTS) + ' products')

# Check whether index already exists
try:
    redis_conn.ft().info()
    print("Index already exists. Dropping old index...")

    redis_conn.ft().dropindex()

except redis.exceptions.ResponseError:
    print("No existing index found.")

# Create flat index
create_flat_index(
    redis_conn,
    ITEM_KEYWORD_EMBEDDING_FIELD,
    NUMBER_PRODUCTS,
    TEXT_EMBEDDING_DIMENSION,
    'COSINE'
)

# Load vectors
load_vectors(
    redis_conn,
    product_metadata,
    item_keywords_vectors,
    ITEM_KEYWORD_EMBEDDING_FIELD
)

print("Products indexed successfully!")

Loading and Indexing + 1000 products
Index already exists. Dropping old index...
Products indexed successfully!


In [54]:
import os

from langchain.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain.chains import LLMChain

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)

prompt = PromptTemplate(
    input_variables=["product_description"],
    template="Create comma separated product keywords to perform a query on an amazon dataset for this user input: {product_description}",
)

chain = LLMChain(
    llm=llm,
    prompt=prompt
)

print("Groq LLM initialized successfully!")

Groq LLM initialized successfully!


In [55]:
userinput = input("Hey im a E-commerce Chatbot, how can i help you today? ")

print("User:", userinput)

# Run the chain only specifying the input variable
keywords = chain.run(userinput)

print("Generated Keywords:", keywords)

User: eneter
Generated Keywords: keyboard, computer keyboard, mechanical keyboard, wireless keyboard, ergonomic keyboard, laptop keyboard, USB keyboard, gaming keyboard, office keyboard, typing accessories, computer accessories, office supplies, input device, keycaps, backlit keyboard, Bluetooth keyboard, portable keyboard, desktop keyboard, typing tool, keyboard cover, keyboard stand, keyboard protector, keyboard wrist rest, keyboard cleaning kit, keyboard case, keyboard pad, keyboard tray, keyboard holder, keyboard dock, keyboard adapter, keyboard connector, keyboard layout, keyboard shortcut guide, keyboard shortcut stickers, keyboard shortcut cheat sheet, keyboard shortcut reference, keyboard shortcut list, keyboard shortcut keys, keyboard shortcut software, keyboard shortcut app, keyboard shortcut tutorial, keyboard shortcut training, keyboard shortcut cheat sheet printable, keyboard shortcut cheat sheet PDF, keyboard shortcut cheat sheet image, keyboard shortcut cheat sheet PDF d

In [56]:
topK = 3

# Vectorize the query
query_vector = model.encode(keywords).astype(np.float32).tobytes()

# Prepare the query
q = Query(
    f'*=>[KNN {topK} @{ITEM_KEYWORD_EMBEDDING_FIELD} $vec_param AS vector_score]'
).sort_by('vector_score').paging(0, topK).return_fields(
    'vector_score',
    'item_name',
    'item_id',
    'item_keywords'
).dialect(2)

params_dict = {"vec_param": query_vector}

# Execute the query
results = redis_conn.ft().search(
    q,
    query_params=params_dict
)

full_result_string = ''

for product in results.docs:
    full_result_string += (
        str(product.item_name) + ' '
        + str(product.item_keywords) + ' '
        + str(product.item_id)
        + "\n\n\n"
    )

print(full_result_string)

Amazon Brand - Solimo Protective Clear View flip Cover for Honor 9X (Blue) phone case phone cover mobile cover mobile case phone flip cover phone cover for honor phone case for honor mobile case for honor mobile case mobile case for honor mobile cover phone case phone case for honor phone cover phone cover for honor phone flip cover mobile case mobile case for honor mobile cover phone case phone case for honor phone cover phone cover for honor phone flip cover mobile case mobile case for honor mobile cover phone case phone case for honor phone cover phone cover for honor phone fli B07VMRRF7J


AmazonBasics Travel and Storage Hard Carrying Case for Bose Soundlink Revolve - Black Bose; accessories; revolve, carry case, carrying case, speaker case, speaker storage case Bose; accessories; revolve, carry case, carrying case, speaker case, speaker storage case Bose; accessories; revolve, carry case, carrying case, speaker case, speaker storage case Bose; accessories; revolve, carry case, car

In [57]:
from langchain.memory import ConversationBufferMemory

template = """You are a chatbot. Be kind, detailed and nice. Present the given queried search result in a nice way as answer to the user input. dont ask questions back! just take the given context

{chat_history}
Human: {user_msg}
Chatbot:"""

prompt = PromptTemplate(
    input_variables=["chat_history", "user_msg"],
    template=template
)

memory = ConversationBufferMemory(
    memory_key="chat_history"
)

llm_chain = LLMChain(
    llm=ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0,
        api_key=os.getenv("GROQ_API_KEY")
    ),
    prompt=prompt,
    verbose=False,
    memory=memory,
)

In [58]:
answer = llm_chain.predict(
    user_msg=f"{full_result_string} ---\n\n {userinput}"
)

print("Bot:", answer)

time.sleep(0.5)

Bot: Here’s a friendly snapshot of the three items you’re looking at on Amazon:

---

### 1. **Solimo Protective Clear View Flip Cover – Blue (for Honor 9X)**
**ASIN:** B07VMRRF7J  

- **Designed for:** Honor 9X – fits perfectly around the edges and button placements.  
- **Style:** Clear front panel lets the phone’s original design shine through, while the blue flip cover adds a pop of color.  
- **Protection:** Flip‑over design shields the screen from scratches and smudges; the back is reinforced to guard against everyday bumps.  
- **Convenient features:**  
  - Magnetic closure keeps the cover securely shut.  
  - Easy access to all ports, speakers, and camera without removing the case.  
- **Why you’ll love it:** Lightweight, slim, and affordable – a great way to keep your Honor 9X safe without sacrificing style.

---

### 2. **AmazonBasics Travel & Storage Hard Carrying Case – Black (for Bose SoundLink Revolve)**
**ASIN:** B07D7RRFRR  

- **Purpose‑built:** Tailored to fit the Bo

In [68]:
follow_up = input("Anything else you want to ask about this topic? ")

print("User:", follow_up)

answer = llm_chain.predict(
    user_msg=follow_up
)

print("Bot:", answer)

time.sleep(0.5)

User: enter
Bot: Here’s the information you asked for, nicely organized:

| Product | Compatible Phone | Key Highlights | Amazon ASIN |
|---|---|---|---|
| **Solimo Designer Printed Mobile Cover (Soft & Flexible Back case)** | Xiaomi Redmi Y3 | Soft TPU material, flexible & slim, transparent with a stylish printed design | **B07R5VF2SJ** |
| **Solimo Designer “Girl Smile” 3D‑Printed Hard Back Case** | Motorola Moto G5 S Plus | Hard polycarbonate, vibrant 3‑D “Girl Smile” graphic, precise cut‑outs for ports | **B08541R35V** |
| **Solimo Designer “Apna Time Ayega” 3D‑Printed Hard Back Case** | Samsung Galaxy J7 Prime | Hard polycarbonate, bold “Apna Time Ayega” artwork, full‑coverage protection | **B0853WZXF8** |

These items are all **mobile‑phone cases**, not smartphones. If you need a phone under ₹20,000, you’ll need to run a new search with that specific query. Happy shopping!


In [69]:
import gradio as gr
import traceback
import numpy as np
import time


def as_text(value):
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    return str(value)


def chat_with_bot(history, user_input):

    history = history or []

    if not user_input or not user_input.strip():
        return history, ""

    try:
        # Generate product keywords
        keywords = chain.run(user_input)

        # Vectorize the query
        query_vector = model.encode(keywords).astype(np.float32).tobytes()

        # Prepare Redis search query
        topK = 3

        q = Query(
            f'*=>[KNN {topK} @{ITEM_KEYWORD_EMBEDDING_FIELD} $vec_param AS vector_score]'
        ).sort_by('vector_score').paging(0, topK).return_fields(
            'vector_score',
            'item_name',
            'item_id',
            'item_keywords'
        ).dialect(2)

        params_dict = {"vec_param": query_vector}

        # Execute Redis search
        results = redis_conn.ft().search(
            q,
            query_params=params_dict
        )

        # Collect product search results
        full_result_string = ""

        for product in results.docs:
            full_result_string += (
                as_text(product.item_name) + " "
                + as_text(product.item_keywords) + " "
                + as_text(product.item_id)
                + "\n\n"
            )

        # Generate chatbot response
        chatbot_input = (
            f"Product search results:\n{full_result_string}\n\n"
            f"User question: {user_input}"
        )

        bot_response = llm_chain.predict(
            user_msg=chatbot_input
        )

        time.sleep(0.5)

        # Update conversation history
        history.append({
            "role": "user",
            "content": user_input
        })

        history.append({
            "role": "assistant",
            "content": bot_response
        })

        return history, ""

    except Exception as e:
        print("Chatbot Error:")
        traceback.print_exc()

        return history, ""


# Create Gradio interface
with gr.Blocks() as demo:

    gr.Markdown("### 💬 AI E-commerce Chatbot")

    chatbot = gr.Chatbot()

    msg = gr.Textbox(
        placeholder="Ask your question here...",
        label="Your Message"
    )

    send = gr.Button("Send")

    send.click(
        chat_with_bot,
        inputs=[chatbot, msg],
        outputs=[chatbot, msg]
    )

    msg.submit(
        chat_with_bot,
        inputs=[chatbot, msg],
        outputs=[chatbot, msg]
    )


# Launch chatbot
demo.launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.
